In [65]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [66]:
from typing import Literal
from pydantic import BaseModel, Field


class GrammarRule(BaseModel):
    id: str = Field(description="Unique identifier, for example a1-present-simple-third-person")
    level: Literal["A1", "A2", "B1"] = Field(description="CEFR learner level")
    category: str = Field(description="General category such as tenses, nouns, articles, or conditionals")
    topic: str = Field(description="Specific grammar topic")
    title: str = Field(description="Short descriptive title for the rule")
    rule: str = Field(description="The grammar rule stated clearly and accurately")
    explanation: str = Field(description="Simple explanation suitable for an English learner")
    correct_example_1: str = Field(description="First grammatically correct example")
    correct_example_2: str = Field(description="Second grammatically correct example")
    incorrect_example: str = Field(description="A grammatically incorrect example")
    corrected_example: str = Field(description="Corrected version of the mistake")
    error_reason: str = Field(description="Simple explanation of why the incorrect example is wrong")
    keywords: str = Field(description="Keywords learners may use when asking about this rule")


class GrammarDataset(BaseModel):
    rules: list[GrammarRule]

In [67]:
prompt = """
Generate exactly 51 English grammar rule cards for adult learners
between CEFR levels A1 and B1.

Requirements:

1. Include a balanced mixture of A1, A2, and B1 rules.
2. Focus on common learner mistakes.
3. Each card must cover one specific grammar rule.
4. Do not create duplicate or strongly overlapping cards.
5. Keep explanations clear and concise.
6. Provide grammatically correct examples.
7. Use British English conventions.
8. Include topics such as:
   - the verb be
   - subject pronouns
   - articles
   - plurals
   - present simple
   - present continuous
   - past simple
   - future forms
   - countable and uncountable nouns
   - comparatives
   - modal verbs
   - present perfect
   - conditionals
   - passive voice
   - gerunds and infinitives
9. Use lowercase, hyphen-separated IDs.
""".strip()


response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    text_format=GrammarDataset,
)

grammar_dataset = response.output_parsed

print(len(grammar_dataset.rules))

64


In [68]:
import pandas as pd


records = [rule.model_dump() for rule in grammar_dataset.rules]
df = pd.DataFrame(records)
df.to_csv("data/grammar_rules.csv",index=False)

In [69]:
df.head()

,id,level,category,topic,title,rule,explanation,correct_example_1,correct_example_2,incorrect_example,corrected_example,error_reason,keywords
0,a1-subject-pronouns-i-you-he-she-it-we-they,A1,pronouns,subject pronouns,Subject pronouns before the verb,"Use subject pronouns such as I, you, he, she, ...",Subject pronouns replace nouns and show who do...,She works in a bank.,They live in London.,Her works in a bank.,She works in a bank.,"Her is an object pronoun, not a subject pronoun.","subject pronouns, i you he she it we they, pro..."
1,a1-verb-be-present,A1,verbs,verb be,Present form of be,"Use am with I, is with he, she, and it, and ar...",The verb be changes with the subject.,I am tired.,They are ready.,She are happy.,She is happy.,"She takes is, not are.","verb be, am is are, be"
2,a1-verb-be-negatives,A1,verbs,verb be,Negative form of be,"Form negatives with am not, is not, or are not.",Add not after the correct form of be.,He is not at home.,We are not late.,I not am hungry.,I am not hungry.,The negative comes after am: am not.,"am not, is not, are not, negative be"
3,a1-verb-be-questions,A1,verbs,verb be,Questions with be,"In questions, put am, is, or are before the su...",Invert the subject and the verb be.,Are you from Spain?,Is he your brother?,You are from Spain?,Are you from Spain?,Questions with be need inversion.,"questions with be, are you, is he, am i"
4,a1-articles-a-an-the,A1,articles,articles,"Using a, an, and the","Use a before consonant sounds, an before vowel...",Choose the article by sound and meaning.,She has a car.,He is an engineer.,I bought an book.,I bought a book.,"Book begins with a consonant sound, so use a.","a an the, articles"


In [70]:
print(df["level"].value_counts())

level
A2    29
B1    20
A1    15
Name: count, dtype: int64


In [71]:
# checking for duplicate IDs and titles
duplicate_ids = df[df["id"].duplicated(keep=False)]
duplicate_ids

duplicate_titles = df[df["title"].duplicated(keep=False)]
duplicate_titles

,id,level,category,topic,title,rule,explanation,correct_example_1,correct_example_2,incorrect_example,corrected_example,error_reason,keywords


In [72]:
import pandas as pd

df = pd.read_csv("data/grammar_rules.csv")
df = df.fillna("")

documents = df.to_dict(orient="records")

In [73]:
documents[0]

{'id': 'a1-subject-pronouns-i-you-he-she-it-we-they',
 'level': 'A1',
 'category': 'pronouns',
 'topic': 'subject pronouns',
 'title': 'Subject pronouns before the verb',
 'rule': 'Use subject pronouns such as I, you, he, she, it, we, and they before the verb.',
 'explanation': 'Subject pronouns replace nouns and show who does the action.',
 'correct_example_1': 'She works in a bank.',
 'correct_example_2': 'They live in London.',
 'incorrect_example': 'Her works in a bank.',
 'corrected_example': 'She works in a bank.',
 'error_reason': 'Her is an object pronoun, not a subject pronoun.',
 'keywords': 'subject pronouns, i you he she it we they, pronouns'}

In [74]:
from minsearch import Index

index = Index(
    text_fields=[
        "title",
        "rule",
        "explanation",
        "correct_example_1",
        "correct_example_2",
        "incorrect_example",
        "corrected_example",
        "error_reason",
        "keywords",
    ],
    keyword_fields=[
        "id",
        "level",
        "category",
        "topic",
    ],
)

index.fit(documents)

In [75]:
def search_grammar_rules(
    query: str,
    level: str | None = None,
    num_results: int = 5,
):
    filter_dict = {}

    if level:
        filter_dict["level"] = level

    boost_dict = {
        "title": 3.0,
        "keywords": 2.5,
        "rule": 2.0,
        "incorrect_example": 2.0,
        "error_reason": 1.5,
        "explanation": 1.0,
    }

    results = index.search(
        query=query,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=num_results,
    )

    return results


In [76]:
results = search_grammar_rules(
    query="Why is 'she go to school' incorrect?",
    level="A1",
)

for result in results:
    print(result["id"])
    print(result["title"])
    print(result["rule"])
    print()

a1-verb-be-present
Present form of be
Use am with I, is with he, she, and it, and are with you, we, and they.

a1-there-is-there-are
There is and there are
Use there is for one thing and there are for more than one.

a1-present-simple-habit
Present simple for habits
Use the present simple for routines, facts, and habits.

a1-subject-pronouns-i-you-he-she-it-we-they
Subject pronouns before the verb
Use subject pronouns such as I, you, he, she, it, we, and they before the verb.

a1-present-simple-third-person
Third person singular present simple
Add -s or -es to the verb with he, she, or it in the present simple.



Additional testing for relevance

In [77]:
results = search_grammar_rules(
    query="When should I use a or an?",
    level="A1",
)

for result in results:
    print(result["title"])

Using a, an, and the
Questions with do and does
Do and does in negatives
Adjectives before nouns
Regular plural nouns


In [78]:
results = search_grammar_rules(
    query="What is the difference between past simple and present perfect?",
    level="B1",
)

for result in results:
    print(result["title"])

Present perfect simple or continuous
Past simple or present perfect in conversation
Passive with present perfect
Third conditional for past regret
Present perfect continuous for duration


In [79]:
prompt_template = """
You are a patient English grammar tutor for adult learners.

Answer the QUESTION using only the grammar information in the CONTEXT.

Rules:

1. Do not invent grammar rules that are absent from the context.
2. Use language appropriate for the learner's stated level.
3. When correcting a sentence, include:
   - the corrected sentence
   - what changed
   - the relevant grammar rule
   - one short practice question
4. Keep the explanation concise.
5. Use British English.
6. When the context does not contain enough information, say that the
   knowledge base does not contain a suitable rule.

LEARNER LEVEL: {level}

QUESTION:
{question}

CONTEXT:
{context}
""".strip()


entry_template = """
Rule ID: {id}
Level: {level}
Category: {category}
Topic: {topic}
Title: {title}
Rule: {rule}
Explanation: {explanation}
Correct example 1: {correct_example_1}
Correct example 2: {correct_example_2}
Incorrect example: {incorrect_example}
Corrected example: {corrected_example}
Reason: {error_reason}
""".strip()

In [80]:
def build_prompt(
    query: str,
    search_results: list[dict],
    level: str,
):
    context_entries = []

    for document in search_results:
        entry = entry_template.format(**document)
        context_entries.append(entry)

    context = "\n\n".join(context_entries)

    prompt = prompt_template.format(
        question=query,
        level=level,
        context=context,
    )

    return prompt

In [81]:
results = search_grammar_rules(
    "Why is 'she go to work' wrong?",
    level="A1",
)

prompt = build_prompt(
    query="Why is 'she go to work' wrong?",
    search_results=results,
    level="A1",
)

print(prompt)

You are a patient English grammar tutor for adult learners.

Answer the QUESTION using only the grammar information in the CONTEXT.

Rules:

1. Do not invent grammar rules that are absent from the context.
2. Use language appropriate for the learner's stated level.
3. When correcting a sentence, include:
   - the corrected sentence
   - what changed
   - the relevant grammar rule
   - one short practice question
4. Keep the explanation concise.
5. Use British English.
6. When the context does not contain enough information, say that the
   knowledge base does not contain a suitable rule.

LEARNER LEVEL: A1

QUESTION:
Why is 'she go to work' wrong?

CONTEXT:
Rule ID: a1-verb-be-present
Level: A1
Category: verbs
Topic: verb be
Title: Present form of be
Rule: Use am with I, is with he, she, and it, and are with you, we, and they.
Explanation: The verb be changes with the subject.
Correct example 1: I am tired.
Correct example 2: They are ready.
Incorrect example: She are happy.
Corrected 

In [82]:
def call_llm(
    prompt: str,
    model: str = "gpt-5.4-mini",
):
    response = openai_client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return response.output_text

Making a RAG function by combining the functions above

In [83]:
def rag(
    query: str,
    level: str = "A1",
    model: str = "gpt-5.4-mini",
):
    search_results = search_grammar_rules(
        query=query,
        level=level,
        num_results=5,
    )

    prompt = build_prompt(
        query=query,
        search_results=search_results,
        level=level,
    )

    answer = call_llm(
        prompt=prompt,
        model=model,
    )

    source_ids = [
        document["id"]
        for document in search_results
    ]

    return {
        "answer": answer,
        "source_ids": source_ids,
    }

In [84]:
result = rag(
    query="Why is 'She go to school every day' incorrect?",
    level="A1",
)

print(result["answer"])
print()
print("Retrieved rules:", result["source_ids"])

The knowledge base does not contain a suitable rule for this sentence.

What I can say from the context:
- **present simple for habits** is used for routines and habits.
- **She go to school every day** is about a routine, so it should use the present simple.

**But** the context does not give a rule for **he/she/it + verb** in the present simple, so I cannot correct it fully from the provided rules.

**Practice question:**  
Is this a routine or a single action? **I play tennis on Saturdays.**

Retrieved rules: ['a1-verb-be-present', 'a1-there-is-there-are', 'a1-present-simple-habit', 'a1-subject-pronouns-i-you-he-she-it-we-they', 'a1-verb-be-questions']


In [85]:
result = rag(
    query="Please explain when to use a and an.",
    level="A1",
)

print(result["answer"])

Use **a** before a **consonant sound** and **an** before a **vowel sound**.

### Examples
- **a car**
- **an engineer**

### What changed?
- We choose **a** or **an** by **sound**, not just by spelling.

### Rule
- Use **a** before consonant sounds.
- Use **an** before vowel sounds.

### Practice question
Choose the correct article: **___ apple**?


In [86]:
result = rag(
    query="I have visited London yesterday. Is this sentence correct?",
    level="B1",
)

print(result["answer"])

No, this sentence is not correct.

**Corrected sentence:**  
**I visited London yesterday.**

**What changed:**  
- **have visited** → **visited**

**Relevant grammar rule:**  
Use the **past simple** for **finished time**. Words like **yesterday** usually take the past simple.

**Practice question:**  
Choose the correct sentence: **She ____ him last week.**


In [87]:
from typing import Literal
from pydantic import BaseModel, Field


class EvaluationQuestion(BaseModel):
    question: str = Field(
        description="A natural question that an English learner might ask"
    )

    query_type: Literal[
        "rule_question",
        "sentence_correction",
        "indirect_question",
    ]


class EvaluationQuestionSet(BaseModel):
    questions: list[EvaluationQuestion]

In [90]:
MODEL = "gpt-5.4-mini"

In [91]:
evaluation_question_prompt = """
You are creating retrieval evaluation questions for an English grammar tutor.

Generate exactly 3 learner questions for the grammar card below.

Requirements:

1. One direct grammar-rule question.
2. One question containing a learner mistake.
3. One indirect or informal question.
4. The questions must be answerable using this grammar card.
5. Do not include the answer.
6. Use natural wording that an adult English learner might use.
7. Do not repeat the title word for word.

Grammar card:

Level: {level}
Topic: {topic}
Title: {title}
Rule: {rule}
Explanation: {explanation}
Incorrect example: {incorrect_example}
Corrected example: {corrected_example}
""".strip()

In [92]:
from tqdm.auto import tqdm
import pandas as pd


ground_truth_rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    prompt = evaluation_question_prompt.format(**row.to_dict())

    response = openai_client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        text_format=EvaluationQuestionSet,
    )

    question_set = response.output_parsed

    for item in question_set.questions:
        ground_truth_rows.append(
            {
                "id": row["id"],
                "level": row["level"],
                "topic": row["topic"],
                "question": item.question,
                "query_type": item.query_type,
            }
        )


df_questions = pd.DataFrame(ground_truth_rows)

print("Evaluation questions:", len(df_questions))
df_questions.head()

100%|██████████| 64/64 [01:55<00:00,  1.81s/it]

Evaluation questions: 192


,id,level,topic,question,query_type
0,a1-subject-pronouns-i-you-he-she-it-we-they,A1,subject pronouns,"When do I use subject pronouns like I, you, he...",rule_question
1,a1-subject-pronouns-i-you-he-she-it-we-they,A1,subject pronouns,Is this sentence correct: Her works in a bank.,sentence_correction
2,a1-subject-pronouns-i-you-he-she-it-we-they,A1,subject pronouns,Can I say 'She works in a bank' instead of usi...,indirect_question
3,a1-verb-be-present,A1,verb be,"When do I use am, is, and are with different s...",rule_question
4,a1-verb-be-present,A1,verb be,She are happy. Is that correct?,sentence_correction


In [ ]:
df_questions.to_csv(
    "data/ground-truth-retrieval.csv",
    index=False,
)

In [94]:
df_questions.sample(10, random_state=1)

,id,level,topic,question,query_type
44,a1-possessive-adjectives,A1,possessive adjectives,How do I say it when I want to show something ...,indirect_question
69,a2-past-simple-negative-question,A2,past simple,How do I make negative sentences and questions...,rule_question
161,b1-gerund-after-prepositions,B1,gerunds and infinitives,Can you explain why we say 'good at speaking' ...,indirect_question
35,a1-adjectives-order,A1,adjective order,"Do we say ""a big dog"" or ""a dog big""?",indirect_question
182,b1-relative-pronouns-who-which-that,B1,relative clauses,"Can I say that for a person, or should I use a...",indirect_question
11,a1-verb-be-questions,A1,verb be,Can I say 'Am I late?' when I ask about myself?,indirect_question
122,a2-passive-present-simple,A2,passive voice,"Can I say ""This language in our office"" when I...",indirect_question
81,a2-superlatives-short-adjectives,A2,superlatives,When do I use -est and when do I use most with...,rule_question
110,a2-present-perfect-vs-past-simple,A2,present perfect versus past simple,"Have you ever tried sushi, or did you try it w...",indirect_question
180,b1-relative-pronouns-who-which-that,B1,relative clauses,"When do I use who, which, and that in a relati...",rule_question


In [95]:
def hit_rate(relevance_total):
    hits = 0

    for relevance in relevance_total:
        if True in relevance:
            hits += 1

    return hits / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0

    for relevance in relevance_total:
        for rank, is_relevant in enumerate(relevance):
            if is_relevant:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

In [96]:
def evaluate_retrieval(
    ground_truth,
    search_function,
):
    relevance_total = []

    for record in ground_truth:
        results = search_function(record)

        relevance = [
            result["id"] == record["id"]
            for result in results
        ]

        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [97]:
unique_ids = (
    pd.Series(df_questions["id"].unique())
    .sample(frac=1, random_state=42)
    .tolist()
)

split_position = int(len(unique_ids) * 0.7)

validation_ids = set(unique_ids[:split_position])
test_ids = set(unique_ids[split_position:])

df_validation = df_questions[
    df_questions["id"].isin(validation_ids)
]

df_test = df_questions[
    df_questions["id"].isin(test_ids)
]

validation_records = df_validation.to_dict(orient="records")
test_records = df_test.to_dict(orient="records")

print("Validation questions:", len(validation_records))
print("Test questions:", len(test_records))

Validation questions: 132
Test questions: 60


In [98]:
def search_with_settings(
    query,
    level=None,
    boosts=None,
    num_results=5,
    use_level_filter=True,
):
    if boosts is None:
        boosts = {}

    filter_dict = {}

    if use_level_filter and level:
        filter_dict["level"] = level

    return index.search(
        query=query,
        filter_dict=filter_dict,
        boost_dict=boosts,
        num_results=num_results,
    )

In [99]:
boost_candidates = {
    "equal": {
        "title": 1.0,
        "rule": 1.0,
        "explanation": 1.0,
        "keywords": 1.0,
        "incorrect_example": 1.0,
        "error_reason": 1.0,
    },

    "topic_focused": {
        "title": 3.0,
        "rule": 2.0,
        "explanation": 1.0,
        "keywords": 3.0,
        "incorrect_example": 1.5,
        "error_reason": 1.0,
    },

    "correction_focused": {
        "title": 2.0,
        "rule": 2.0,
        "explanation": 1.0,
        "keywords": 2.5,
        "incorrect_example": 3.0,
        "error_reason": 2.0,
    },

    "rule_focused": {
        "title": 2.0,
        "rule": 4.0,
        "explanation": 2.0,
        "keywords": 2.0,
        "incorrect_example": 1.0,
        "error_reason": 1.0,
    },
}

In [100]:
retrieval_experiments = []

for boost_name, boosts in boost_candidates.items():
    for top_k in [3, 5, 10]:
        for use_level_filter in [True, False]:

            def experiment_search(record):
                return search_with_settings(
                    query=record["question"],
                    level=record["level"],
                    boosts=boosts,
                    num_results=top_k,
                    use_level_filter=use_level_filter,
                )

            scores = evaluate_retrieval(
                validation_records,
                experiment_search,
            )

            retrieval_experiments.append(
                {
                    "boost_name": boost_name,
                    "top_k": top_k,
                    "use_level_filter": use_level_filter,
                    "hit_rate": scores["hit_rate"],
                    "mrr": scores["mrr"],
                }
            )


df_retrieval_experiments = pd.DataFrame(
    retrieval_experiments
)

df_retrieval_experiments.sort_values(
    ["mrr", "hit_rate"],
    ascending=False,
).head(10)

,boost_name,top_k,use_level_filter,hit_rate,mrr
16,correction_focused,10,True,0.962121,0.801659
14,correction_focused,5,True,0.946970,0.799495
4,equal,10,True,0.977273,0.793978
12,correction_focused,3,True,0.893939,0.786616
2,equal,5,True,0.924242,0.786237
22,rule_focused,10,True,0.977273,0.785414
10,topic_focused,10,True,0.962121,0.784500
8,topic_focused,5,True,0.946970,0.782576
20,rule_focused,5,True,0.931818,0.779545
0,equal,3,True,0.878788,0.775253


In [101]:
best_row = (
    df_retrieval_experiments
    .sort_values(
        ["mrr", "hit_rate"],
        ascending=False,
    )
    .iloc[0]
)

best_boost_name = best_row["boost_name"]
best_boosts = boost_candidates[best_boost_name]
best_top_k = int(best_row["top_k"])
best_use_level_filter = bool(
    best_row["use_level_filter"]
)

print("Best boost configuration:", best_boost_name)
print("Best top-k:", best_top_k)
print("Use level filter:", best_use_level_filter)
print("Boost values:", best_boosts)

Best boost configuration: correction_focused
Best top-k: 10
Use level filter: True
Boost values: {'title': 2.0, 'rule': 2.0, 'explanation': 1.0, 'keywords': 2.5, 'incorrect_example': 3.0, 'error_reason': 2.0}


In [102]:
def final_keyword_search(record):
    return search_with_settings(
        query=record["question"],
        level=record["level"],
        boosts=best_boosts,
        num_results=best_top_k,
        use_level_filter=best_use_level_filter,
    )


keyword_test_scores = evaluate_retrieval(
    test_records,
    final_keyword_search,
)

keyword_test_scores

{'hit_rate': 0.95, 'mrr': 0.7805555555555556}

In [103]:
df_retrieval_experiments.to_csv(
    "data/retrieval-keyword-experiments.csv",
    index=False,
)

Comparing vector search and hybrid search


In [104]:
def grammar_card_to_text(document):
    return f"""
Title: {document["title"]}
Level: {document["level"]}
Category: {document["category"]}
Topic: {document["topic"]}
Rule: {document["rule"]}
Explanation: {document["explanation"]}
Correct example: {document["correct_example_1"]}
Correct example: {document["correct_example_2"]}
Incorrect example: {document["incorrect_example"]}
Corrected example: {document["corrected_example"]}
Error reason: {document["error_reason"]}
Keywords: {document["keywords"]}
""".strip()

In [105]:
document_texts = [
    grammar_card_to_text(document)
    for document in documents
]

print(document_texts[0])

Title: Subject pronouns before the verb
Level: A1
Category: pronouns
Topic: subject pronouns
Rule: Use subject pronouns such as I, you, he, she, it, we, and they before the verb.
Explanation: Subject pronouns replace nouns and show who does the action.
Correct example: She works in a bank.
Correct example: They live in London.
Incorrect example: Her works in a bank.
Corrected example: She works in a bank.
Error reason: Her is an object pronoun, not a subject pronoun.
Keywords: subject pronouns, i you he she it we they, pronouns


In [108]:
import numpy as np


embedding_response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=document_texts,
)

document_vectors = np.array(
    [
        item.embedding
        for item in embedding_response.data
    ],
    dtype=np.float32,
)

print("Vector matrix shape:", document_vectors.shape)

Vector matrix shape: (64, 1536)


In [111]:
from minsearch import VectorSearch


vector_index = VectorSearch(
    keyword_fields=[
        "level",
        "category",
        "topic",
    ]
)

vector_index.fit(
    document_vectors,
    documents,
)

In [112]:
def embed_query(query):
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query,
    )

    return np.array(
        response.data[0].embedding,
        dtype=np.float32,
    )

In [113]:
def vector_search(
    query,
    level=None,
    num_results=5,
    use_level_filter=True,
):
    query_vector = embed_query(query)

    filter_dict = {}

    if use_level_filter and level:
        filter_dict["level"] = level

    return vector_index.search(
        query_vector,
        filter_dict=filter_dict,
        num_results=num_results,
    )

In [114]:
results = vector_search(
    query="Why does the verb change after she?",
    level="A1",
)

for result in results:
    print(result["id"], "-", result["title"])

a1-present-simple-third-person - Third person singular present simple
a1-subject-pronouns-i-you-he-she-it-we-they - Subject pronouns before the verb
a1-verb-be-present - Present form of be
a1-plurals-spelling-changes - Plural spelling changes
a1-verb-be-questions - Questions with be


In [115]:
evaluation_questions = (
    df_questions["question"]
    .drop_duplicates()
    .tolist()
)

evaluation_embedding_response = (
    openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=evaluation_questions,
    )
)

evaluation_vectors = np.array(
    [
        item.embedding
        for item in evaluation_embedding_response.data
    ],
    dtype=np.float32,
)

question_vector_lookup = {
    question: vector
    for question, vector in zip(
        evaluation_questions,
        evaluation_vectors,
    )
}

np.save(
    "data/evaluation-question-embeddings.npy",
    evaluation_vectors,
)

In [116]:
def vector_search_for_evaluation(
    record,
    num_results=5,
    use_level_filter=True,
):
    query_vector = question_vector_lookup[
        record["question"]
    ]

    filter_dict = {}

    if use_level_filter:
        filter_dict["level"] = record["level"]

    return vector_index.search(
        query_vector,
        filter_dict=filter_dict,
        num_results=num_results,
    )

Creating hybrid search

In [117]:
def reciprocal_rank_fusion(
    result_lists,
    rrf_constant=60,
    num_results=5,
):
    scores = {}
    documents_by_id = {}

    for results in result_lists:
        for rank, document in enumerate(
            results,
            start=1,
        ):
            document_id = document["id"]

            documents_by_id[document_id] = document

            scores[document_id] = (
                scores.get(document_id, 0.0)
                + 1 / (rrf_constant + rank)
            )

    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True,
    )

    return [
        documents_by_id[document_id]
        for document_id in ranked_ids[:num_results]
    ]

In [118]:
def hybrid_search_for_evaluation(
    record,
    num_results=5,
):
    text_results = search_with_settings(
        query=record["question"],
        level=record["level"],
        boosts=best_boosts,
        num_results=10,
        use_level_filter=best_use_level_filter,
    )

    vector_results = vector_search_for_evaluation(
        record,
        num_results=10,
        use_level_filter=best_use_level_filter,
    )

    return reciprocal_rank_fusion(
        [text_results, vector_results],
        num_results=num_results,
    )

In [119]:
def hybrid_search_for_evaluation(
    record,
    num_results=5,
):
    text_results = search_with_settings(
        query=record["question"],
        level=record["level"],
        boosts=best_boosts,
        num_results=10,
        use_level_filter=best_use_level_filter,
    )

    vector_results = vector_search_for_evaluation(
        record,
        num_results=10,
        use_level_filter=best_use_level_filter,
    )

    return reciprocal_rank_fusion(
        [text_results, vector_results],
        num_results=num_results,
    )

In [120]:
retrieval_method_results = []

for method_name, search_function in [
    (
        "keyword",
        final_keyword_search,
    ),
    (
        "vector",
        lambda record: vector_search_for_evaluation(
            record,
            num_results=best_top_k,
            use_level_filter=best_use_level_filter,
        ),
    ),
    (
        "hybrid",
        lambda record: hybrid_search_for_evaluation(
            record,
            num_results=best_top_k,
        ),
    ),
]:
    scores = evaluate_retrieval(
        validation_records,
        search_function,
    )

    retrieval_method_results.append(
        {
            "method": method_name,
            "hit_rate": scores["hit_rate"],
            "mrr": scores["mrr"],
        }
    )


df_method_results = pd.DataFrame(
    retrieval_method_results
)

df_method_results.sort_values(
    "mrr",
    ascending=False,
)

,method,hit_rate,mrr
1,vector,1.000000,0.936490
2,hybrid,1.000000,0.894357
0,keyword,0.962121,0.801659


In [121]:
best_method = (
    df_method_results
    .sort_values(
        ["mrr", "hit_rate"],
        ascending=False,
    )
    .iloc[0]["method"]
)

print("Selected retrieval method:", best_method)

Selected retrieval method: vector


In [122]:
df_method_results.to_csv(
    "data/retrieval-method-comparison.csv",
    index=False,
)

LLM as a judge

In [123]:
DIRECT_PROMPT = """
You are an English grammar tutor.

Answer using only the CONTEXT.

Give:

1. The answer or correction.
2. The relevant grammar rule.
3. One brief example.

Use British English.
Use language suitable for level {level}.

QUESTION:
{question}

CONTEXT:
{context}
""".strip()

In [124]:
GUIDED_PROMPT = """
You are a patient English grammar tutor for adult learners.

Answer using only the CONTEXT.

When the learner makes a mistake:

1. Show the corrected sentence.
2. Explain exactly what changed.
3. State the relevant rule simply.
4. Give one additional example.
5. Give one short practice question.

Do not provide unrelated grammar information.
Use British English.
Use language suitable for level {level}.

QUESTION:
{question}

CONTEXT:
{context}
""".strip()

In [125]:
def build_prompt(
    query,
    search_results,
    level,
    prompt_style="guided",
):
    context_entries = [
        entry_template.format(**document)
        for document in search_results
    ]

    context = "\n\n".join(context_entries)

    if prompt_style == "direct":
        template = DIRECT_PROMPT
    else:
        template = GUIDED_PROMPT

    return template.format(
        question=query,
        level=level,
        context=context,
    )

In [126]:
def rag(
    query,
    level="A1",
    model=MODEL,
    prompt_style="guided",
):
    search_results = search_grammar_rules(
        query=query,
        level=level,
        num_results=best_top_k,
    )

    prompt = build_prompt(
        query=query,
        search_results=search_results,
        level=level,
        prompt_style=prompt_style,
    )

    answer = call_llm(
        prompt=prompt,
        model=model,
    )

    return {
        "answer": answer,
        "source_ids": [
            document["id"]
            for document in search_results
        ],
        "search_results": search_results,
    }

In [127]:
class AnswerJudgement(BaseModel):
    grammar_correctness: Literal[
        "PASS",
        "PARTIAL",
        "FAIL",
    ]

    groundedness: Literal[
        "PASS",
        "PARTIAL",
        "FAIL",
    ]

    level_appropriateness: Literal[
        "PASS",
        "PARTIAL",
        "FAIL",
    ]

    helpfulness: Literal[
        "PASS",
        "PARTIAL",
        "FAIL",
    ]

    explanation: str

In [128]:
answer_judge_prompt = """
You are evaluating an English grammar tutoring answer.

Evaluate the answer using these criteria:

1. Grammar correctness:
   Is the correction and explanation grammatically accurate?

2. Groundedness:
   Is the answer supported by the retrieved grammar context?

3. Level appropriateness:
   Is the language suitable for the learner's stated CEFR level?

4. Helpfulness:
   Does the answer clearly address the learner's question?

Use:
PASS
PARTIAL
FAIL

Learner level:
{level}

Question:
{question}

Retrieved context:
{context}

Tutor answer:
{answer}
""".strip()

In [129]:
answer_test_sample = (
    df_test
    .sample(
        n=min(10, len(df_test)),
        random_state=42,
    )
    .to_dict(orient="records")
)

In [130]:
answer_evaluations = []

for record in tqdm(answer_test_sample):
    for prompt_style in ["direct", "guided"]:

        rag_result = rag(
            query=record["question"],
            level=record["level"],
            prompt_style=prompt_style,
        )

        context = "\n\n".join(
            entry_template.format(**document)
            for document in rag_result[
                "search_results"
            ]
        )

        judge_prompt = answer_judge_prompt.format(
            level=record["level"],
            question=record["question"],
            context=context,
            answer=rag_result["answer"],
        )

        judge_response = (
            openai_client.responses.parse(
                model=MODEL,
                input=[
                    {
                        "role": "user",
                        "content": judge_prompt,
                    }
                ],
                text_format=AnswerJudgement,
            )
        )

        judgement = judge_response.output_parsed

        answer_evaluations.append(
            {
                "question": record["question"],
                "expected_id": record["id"],
                "level": record["level"],
                "prompt_style": prompt_style,
                "answer": rag_result["answer"],
                "source_ids": ",".join(
                    rag_result["source_ids"]
                ),
                **judgement.model_dump(),
            }
        )

100%|██████████| 10/10 [01:10<00:00,  7.05s/it]


In [131]:
df_answer_evaluations = pd.DataFrame(
    answer_evaluations
)

for column in [
    "grammar_correctness",
    "groundedness",
    "level_appropriateness",
    "helpfulness",
]:
    print(f"\n{column}")
    print(
        pd.crosstab(
            df_answer_evaluations[
                "prompt_style"
            ],
            df_answer_evaluations[column],
            normalize="index",
        )
    )


grammar_correctness
grammar_correctness  PASS
prompt_style             
direct                1.0
guided                1.0

groundedness
groundedness  PASS
prompt_style      
direct         1.0
guided         1.0

level_appropriateness
level_appropriateness  PASS
prompt_style               
direct                  1.0
guided                  1.0

helpfulness
helpfulness   PASS
prompt_style      
direct         1.0
guided         1.0
